# Snow amount at the start of spring and the runoff onset anomaly

Does the snow on the ground when the melt season begins pace runoff onset, alongside spring air temperature? A deeper
pack holds more cold content and takes more energy to warm and wet, so with the same weather a deep-snow year should
release runoff later (a positive correlation between snow amount and the onset anomaly). Where energy is abundant at
the time of melt (high latitudes, melt near the solstice), snow amount may matter as much as temperature.

**Snow amount** is the ERA5-Land snow water equivalent anomaly in the first month of the spring window (March in the
Northern Hemisphere, September in the Southern), the weighted zonal mean over each range's analysed snow from the
aggregation cube (`snow_depth_water_equivalent`, `month = spring_month_1`). ERA5-Land SWE is modelled (reanalysis
forcing and snowfall), not observed. **Spring temperature** and the **onset anomaly** are the per-year columns of the
statistics table (spring = March–May / September–November).

Per included range with at least `MIN_YEARS` paired years (the rule every temperature-sensitivity number uses):

- Pearson r of the onset anomaly with spring temperature (`r_T`) and with SWE (`r_SWE`), each one's partial r given
  the other, and r(T, SWE);
- the **Spearman twin** of each (rank correlations; partial Spearman = partial correlation of the ranks);
- by latitude band: medians, the share of ranges where SWE explains more variance than temperature
  (`r_SWE² > r_T²`), and counts of individually significant correlations (p < 0.05);
- poleward of 60°: the count of ranges with a significantly positive SWE correlation against a **shared-year
  permutation null** (one permutation of the water-year labels of the SWE series, the same for every range, which keeps
  the spatial dependence between ranges; the scheme of `range_metrics.field_significance`), for Pearson and Spearman;
- robustness: range-years with coverage ≥ 30 % only, and leave-one-year-out medians.

```
reads                                                          writes (results/<version>/)
─────                                                          ──────────────────────────
results/<v>/mountain_range_statistics.csv ─────────────┐       snow_amount_by_range.csv
data/aggregation/<v>/all_mountain_ranges_fcf_lte_50.nc ┴──►    snow_amount_by_band.csv
                                                               snow_amount_summary.json
```

Run `range_statistics.ipynb` first. Counts and medians of per-range values only; no pooled fits.

In [ ]:
import json

import numpy as np
import pandas as pd
import scipy.stats as sps
import xarray as xr

from gsro_analysis import paths, range_metrics as rm, results, settings

In [ ]:
config = settings.load_config()
VERSION = config.version
WATER_YEARS = [int(y) for y in config.water_years]
UNIT = 'mountain_ranges'
results_dir = paths.resultsdir(UNIT, VERSION)
cube_path = paths.aggregation_dir(UNIT, VERSION) / 'all_mountain_ranges_fcf_lte_50.nc'

MIN_YEARS = 8                                   # paired years per range, as for every temperature-sensitivity number
MIN_COVERAGE_ROBUST = 0.30                      # robustness: range-years with at least this coverage
POLAR_LAT = 60                                  # |centroid latitude| for the high-latitude test
BANDS = [0, 35, 50, 60, 90]                     # |latitude| bands of the band summary
N_PERM = 2000
SEED = 0
ALPHA = 0.05

In [ ]:
statistics_df = pd.read_csv(results_dir / 'mountain_range_statistics.csv').set_index('name')
included = statistics_df.index[statistics_df['included'].astype(bool)]
cols = lambda prefix: [f'{prefix}_WY{y}' for y in WATER_YEARS]
anomaly_df = statistics_df[cols('runoff_onset_anomaly')].set_axis(WATER_YEARS, axis=1)
temperature_df = statistics_df[cols('spring_temperature_anomaly')].set_axis(WATER_YEARS, axis=1)
coverage_df = statistics_df[cols('coverage')].set_axis(WATER_YEARS, axis=1)

cube = xr.open_dataset(cube_path)
swe = cube['snow_depth_water_equivalent'].sel(month='spring_month_1')      # anomaly, m w.e., (mountain_range, water_year)
print(swe.attrs['long_name'], '|', swe.attrs['units'])
swe_df = swe.to_pandas() * 1000                                              # mm w.e.
swe_df.index = swe_df.index.astype(str)
swe_df = swe_df.reindex(columns=WATER_YEARS)
print(f'{len(included)} included ranges; SWE series for {int(swe_df.reindex(included).notna().any(axis=1).sum())} of them')

## 1. Per-range correlations

A range enters when it has at least `MIN_YEARS` water years with an onset anomaly, a spring temperature and a SWE
value. `min_coverage` (default: none beyond the statistics table's own rule) drops range-years with less coverage.

In [ ]:
def partial_r(y, x, z):
    # correlation of x and y after removing the linear effect of z from each
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    ry = y - np.polyval(np.polyfit(z, y, 1), z)
    return float(np.corrcoef(rx, ry)[0, 1])


def per_range_table(min_coverage=0.0):
    rows = {}
    for name in included:
        if name not in swe_df.index:
            continue
        d = pd.DataFrame({'a': anomaly_df.loc[name], 't': temperature_df.loc[name], 's': swe_df.loc[name],
                          'c': coverage_df.loc[name]}).dropna()
        d = d[d['c'] >= min_coverage]
        if len(d) < MIN_YEARS:
            continue
        a, t, s = d['a'].values, d['t'].values, d['s'].values
        ra, rt, rs = sps.rankdata(a), sps.rankdata(t), sps.rankdata(s)
        rows[name] = dict(
            n_years=len(d), centroid_latitude=float(statistics_df.loc[name, 'centroid_latitude']),
            r_T=sps.pearsonr(t, a).statistic, p_T=sps.pearsonr(t, a).pvalue,
            r_SWE=sps.pearsonr(s, a).statistic, p_SWE=sps.pearsonr(s, a).pvalue,
            partial_r_SWE_given_T=partial_r(a, s, t), partial_r_T_given_SWE=partial_r(a, t, s),
            r_T_SWE=sps.pearsonr(t, s).statistic,
            rho_T=sps.spearmanr(t, a).statistic, p_rho_T=sps.spearmanr(t, a).pvalue,
            rho_SWE=sps.spearmanr(s, a).statistic, p_rho_SWE=sps.spearmanr(s, a).pvalue,
            partial_rho_SWE_given_T=partial_r(ra, rs, rt), partial_rho_T_given_SWE=partial_r(ra, rt, rs),
            rho_T_SWE=sps.spearmanr(t, s).statistic)
    df = pd.DataFrame.from_dict(rows, orient='index')
    df.index.name = 'name'
    df['lat_band'] = pd.cut(df['centroid_latitude'].abs(), BANDS).astype(str)
    df['swe_beats_T'] = df['r_SWE'] ** 2 > df['r_T'] ** 2
    return df


by_range_df = per_range_table()
robust_df = per_range_table(MIN_COVERAGE_ROBUST)
print(f'{len(by_range_df)} included ranges with >= {MIN_YEARS} paired years; {len(robust_df)} with coverage >= {MIN_COVERAGE_ROBUST:.0%}')

## 2. By latitude band

Medians of the per-range values, the share of ranges where SWE explains more of the onset variance than spring
temperature, and the counts of individually significant correlations (positive for SWE, negative for temperature).

In [ ]:
def band_summary(df):
    g = df.groupby('lat_band', observed=True, sort=False)
    out = pd.DataFrame({
        'ranges': g.size(),
        'median_r_T': g['r_T'].median(), 'median_r_SWE': g['r_SWE'].median(),
        'median_r2_T': g['r_T'].apply(lambda v: (v ** 2).median()), 'median_r2_SWE': g['r_SWE'].apply(lambda v: (v ** 2).median()),
        'share_swe_beats_T': g['swe_beats_T'].mean(),
        'share_r_SWE_positive': g['r_SWE'].apply(lambda v: (v > 0).mean()),
        'sig_SWE_positive': g.apply(lambda x: int(((x['p_SWE'] < ALPHA) & (x['r_SWE'] > 0)).sum()), include_groups=False),
        'sig_T_negative': g.apply(lambda x: int(((x['p_T'] < ALPHA) & (x['r_T'] < 0)).sum()), include_groups=False),
        'median_partial_r_SWE_given_T': g['partial_r_SWE_given_T'].median(),
        'median_partial_r_T_given_SWE': g['partial_r_T_given_SWE'].median(),
        'median_r_T_SWE': g['r_T_SWE'].median(),
        'median_rho_T': g['rho_T'].median(), 'median_rho_SWE': g['rho_SWE'].median(),
        'sig_rho_SWE_positive': g.apply(lambda x: int(((x['p_rho_SWE'] < ALPHA) & (x['rho_SWE'] > 0)).sum()), include_groups=False),
        'median_partial_rho_SWE_given_T': g['partial_rho_SWE_given_T'].median(),
        'median_rho_T_SWE': g['rho_T_SWE'].median()})
    order = [str(b) for b in pd.cut(pd.Series([10, 40, 55, 70]), BANDS).cat.categories]
    return out.reindex([b for b in order if b in out.index])


band_df = pd.concat({'all_range_years': band_summary(by_range_df), f'coverage_ge_{int(MIN_COVERAGE_ROBUST * 100)}pct': band_summary(robust_df)},
                    names=['selection', 'lat_band'])
band_df.round(2)

## 3. Poleward of 60°: shared-year permutation null

The count of ranges with a significantly positive SWE correlation, against the counts obtained when the water-year
labels of the SWE series are permuted once and the same permutation is applied to every range (repeated `N_PERM`
times). With Pearson r, a significantly positive correlation with SWE is a significantly negative OLS slope on −SWE, so
`range_metrics.field_significance` gives the same count; it is run as a check.

In [ ]:
polar = by_range_df.index[by_range_df['centroid_latitude'].abs() >= POLAR_LAT]


def count_sig_positive(x_df, names, method):
    k = 0
    for name in names:
        ok = anomaly_df.loc[name].notna().values & x_df.loc[name].notna().values
        if ok.sum() < MIN_YEARS:
            continue
        x, y = x_df.loc[name].values[ok], anomaly_df.loc[name].values[ok]
        res = sps.pearsonr(x, y) if method == 'pearson' else sps.spearmanr(x, y)
        k += (res.pvalue < ALPHA) and (res.statistic > 0)
    return int(k)


def shared_year_null(names, method, n_perm=N_PERM, seed=SEED):
    rng = np.random.default_rng(seed)
    x_df = swe_df.reindex(names)
    obs = count_sig_positive(x_df, names, method)
    null = np.array([count_sig_positive(x_df.set_axis(list(rng.permutation(WATER_YEARS)), axis=1)[WATER_YEARS], names, method)
                     for _ in range(n_perm)])
    return {'ranges': len(names), 'obs_sig_positive': obs, 'null_p95': float(np.percentile(null, 95)), 'null_max': int(null.max()),
            'p_value': float((np.sum(null >= obs) + 1) / (n_perm + 1)), 'n_perm': n_perm, 'seed': seed}


polar_null = {'pearson': shared_year_null(polar, 'pearson'), 'spearman': shared_year_null(polar, 'spearman')}
check = rm.field_significance(anomaly_df, -swe_df.reindex(columns=WATER_YEARS), list(polar), n_perm=N_PERM, seed=SEED, min_years=MIN_YEARS)
assert check['obs_sig_neg'] == polar_null['pearson']['obs_sig_positive']
polar_null['pearson_field_significance_check'] = {k: check[k] for k in ('obs_sig_neg', 'null_sig_neg_p95', 'null_sig_neg_max')}
polar_null

## 4. Robustness poleward of 60°: leave one year out

The median r_SWE across the high-latitude ranges, recomputed with each water year left out in turn (a range needs
`MIN_YEARS - 1` years left).

In [ ]:
leave_one_out = {}
for y in WATER_YEARS:
    vals = []
    for name in polar:
        d = pd.DataFrame({'a': anomaly_df.loc[name], 's': swe_df.loc[name]}).dropna().drop(index=y, errors='ignore')
        if len(d) >= MIN_YEARS - 1:
            vals.append(np.corrcoef(d['a'], d['s'])[0, 1])
    leave_one_out[y] = float(np.median(vals))
pd.Series(leave_one_out, name='median_r_SWE_without_year').round(2)

## 5. Summary and outputs

In [ ]:
p = by_range_df.loc[polar]
summary = {
    'min_years': MIN_YEARS, 'polar_lat': POLAR_LAT,
    'ranges_with_min_years': int(len(by_range_df)),
    'polar': {
        'ranges': int(len(p)), 'r_SWE_positive': int((p['r_SWE'] > 0).sum()),
        'median_r_SWE': float(p['r_SWE'].median()), 'median_rho_SWE': float(p['rho_SWE'].median()),
        'sig_SWE_positive': int(((p['p_SWE'] < ALPHA) & (p['r_SWE'] > 0)).sum()),
        'sig_rho_SWE_positive': int(((p['p_rho_SWE'] < ALPHA) & (p['rho_SWE'] > 0)).sum()),
        'median_partial_r_SWE_given_T': float(p['partial_r_SWE_given_T'].median()),
        'median_partial_rho_SWE_given_T': float(p['partial_rho_SWE_given_T'].median()),
        'median_r_T_SWE': float(p['r_T_SWE'].median()), 'median_rho_T_SWE': float(p['rho_T_SWE'].median()),
        'median_r_T': float(p['r_T'].median()),
        'share_swe_beats_T': float(p['swe_beats_T'].mean()),
        'shared_year_null': polar_null,
        'leave_one_year_out_median_r_SWE': {'min': min(leave_one_out.values()), 'max': max(leave_one_out.values())},
        'coverage_ge_30pct': {'ranges': int((robust_df['centroid_latitude'].abs() >= POLAR_LAT).sum()),
                              'median_r_SWE': float(robust_df.loc[robust_df['centroid_latitude'].abs() >= POLAR_LAT, 'r_SWE'].median())}},
    '_provenance': results.provenance(config)}
by_range_df.assign(**results.provenance(config)).to_csv(results_dir / 'snow_amount_by_range.csv', float_format='%.4g')
band_df.assign(**results.provenance(config)).to_csv(results_dir / 'snow_amount_by_band.csv', float_format='%.4g')
with open(results_dir / 'snow_amount_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=float)
print(json.dumps({k: v for k, v in summary.items() if k != '_provenance'}, indent=1, default=float))